# Step 04B — Diagnostic et correction des limites du Deep Learning

Ce notebook ne remplace pas le Step 04.  
Il sert à **diagnostiquer pourquoi le Deep Learning simple ne progresse pas assez** et à tester une version corrigée du pipeline.

L’objectif n’est pas seulement d’entraîner un modèle plus complexe, mais de vérifier méthodiquement :

1. si le texte utilisé pour le Deep Learning est trop nettoyé ;
2. si les textes trop courts ou trop longs perturbent l’apprentissage ;
3. si la tokenisation est adaptée ;
4. si le déséquilibre est bien géré ;
5. si un modèle BiLSTM optimisé améliore les classes faibles ;
6. si le problème vient plutôt du dataset, des labels ou de l’architecture.

La logique suivie ici est :

```text
Diagnostic → correction du texte → tokenisation → modèle corrigé → comparaison → analyse d’erreurs
```

## 1. Installation et importation des librairies

Cette cellule prépare l’environnement.  
Si TensorFlow n’est pas installé, il faut l’installer dans l’environnement Anaconda utilisé par Jupyter.

```bash
pip install tensorflow
```

Les autres librairies sont utilisées pour :
- manipuler les données ;
- construire les séquences textuelles ;
- entraîner le modèle ;
- afficher les métriques.

In [ ]:
import os
import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import (
        Embedding, Dense, Dropout, SpatialDropout1D,
        Bidirectional, LSTM, GlobalMaxPooling1D, Conv1D
    )
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
    from tensorflow.keras.regularizers import l2

    tf.random.set_seed(SEED)
    print("TensorFlow version :", tf.__version__)
except Exception as e:
    raise ImportError(
        "TensorFlow n'est pas disponible. Installez-le avec : pip install tensorflow\n"
        f"Erreur : {e}"
    )

## 2. Chargement du dataset préparé

On charge le dataset issu du Step 02.

Le notebook cherche d’abord `mental_health_cleaned.csv`, car c’est le fichier attendu après le cleaning.  
Si ce fichier n’existe pas, il affiche un message clair.

Important : pour le Deep Learning, on va comparer deux sources de texte si possible :

- `clean_text` : version nettoyée du Step 02 ;
- `statement` : texte brut, si disponible, pour créer une version **light cleaning** moins agressive.

Cette comparaison est essentielle car les modèles séquentiels peuvent perdre de l’information si le texte a été trop simplifié.

In [ ]:
DATA_PATH = "mental_health_cleaned.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "Le fichier mental_health_cleaned.csv est introuvable. "
        "Exécutez d'abord le Step 02 ou placez le fichier dans le même dossier que ce notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset chargé :", df.shape)
print("Colonnes disponibles :", df.columns.tolist())
display(df.head())

## 3. Vérification des colonnes texte et label

On vérifie que :
- la colonne cible est bien `status` ;
- la colonne `clean_text` existe ;
- les textes vides sont retirés uniquement pour la modélisation.

Cette étape évite une erreur fréquente : entraîner le modèle sur une colonne vide ou mal choisie.

In [ ]:
LABEL_COL = "status"

if LABEL_COL not in df.columns:
    raise ValueError("La colonne label 'status' est introuvable.")

if "clean_text" not in df.columns:
    raise ValueError("La colonne 'clean_text' est introuvable. Le Step 02 doit créer cette colonne.")

df_model = df.copy()
df_model[LABEL_COL] = df_model[LABEL_COL].fillna("").astype(str).str.strip()
df_model["clean_text"] = df_model["clean_text"].fillna("").astype(str).str.strip()

df_model = df_model[df_model[LABEL_COL] != ""].copy()
df_model = df_model[df_model["clean_text"] != ""].copy()

df_model["clean_num_words"] = df_model["clean_text"].apply(lambda x: len(x.split()))

print("Dataset après suppression des textes vides :", df_model.shape)
print("Nombre de classes :", df_model[LABEL_COL].nunique())
display(df_model[LABEL_COL].value_counts())
display(df_model["clean_num_words"].describe().round(2))

## 4. Diagnostic : est-ce que le texte nettoyé est trop réduit ?

Si la longueur moyenne ou médiane est trop faible, le Deep Learning peut perdre du contexte.

Un modèle BiLSTM a besoin de séquences suffisamment riches pour apprendre :
- l’ordre des mots ;
- les relations entre mots ;
- les nuances émotionnelles.

Cette cellule mesure la proportion de textes très courts.

In [ ]:
short_threshold = 5

short_rate = (df_model["clean_num_words"] < short_threshold).mean() * 100

print(f"Pourcentage de textes avec moins de {short_threshold} mots :", round(short_rate, 2), "%")

plt.figure(figsize=(10, 5))
plt.hist(df_model["clean_num_words"], bins=80)
plt.xlim(0, 300)
plt.title("Distribution des longueurs des textes nettoyés")
plt.xlabel("Nombre de mots")
plt.ylabel("Nombre de textes")
plt.show()

print("Exemples de textes très courts :")
display(
    df_model.loc[df_model["clean_num_words"] < short_threshold, ["clean_text", LABEL_COL]]
    .head(10)
)

## 5. Création d’une version texte plus légère pour le Deep Learning

Les modèles classiques TF-IDF bénéficient souvent d’un nettoyage fort.  
Les modèles Deep Learning, eux, peuvent bénéficier d’un texte moins agressif, car ils exploitent l’ordre et le contexte.

Si la colonne `statement` est disponible, on crée une colonne `dl_text_light` à partir du texte brut avec un nettoyage léger :

- correction basique des espaces ;
- suppression des URLs et mentions ;
- conservation des hashtags sous forme de mots ;
- conservation des négations ;
- conservation d’une partie de la ponctuation émotionnelle ;
- pas de suppression massive des stopwords.

Si `statement` n’existe pas, on utilise `clean_text`.

In [ ]:
def light_clean_for_deep_learning(text):
    text = str(text)
    text = text.lower()

    # URLs et mentions : peu utiles pour la sémantique
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)

    # Hashtags : garder le mot sans #
    text = re.sub(r"#(\w+)", r"\1", text)

    # Garder lettres, chiffres, apostrophes et ponctuation émotionnelle simple
    text = re.sub(r"[^a-z0-9\s'!?]", " ", text)

    # Réduire ponctuation excessive sans la supprimer totalement
    text = re.sub(r"!{2,}", " ! ", text)
    text = re.sub(r"\?{2,}", " ? ", text)

    # Réduire espaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

if "statement" in df_model.columns:
    df_model["dl_text_light"] = df_model["statement"].fillna("").astype(str).apply(light_clean_for_deep_learning)
    TEXT_DL_COL = "dl_text_light"
    print("Version Deep Learning créée à partir de 'statement' avec nettoyage léger.")
else:
    df_model["dl_text_light"] = df_model["clean_text"]
    TEXT_DL_COL = "dl_text_light"
    print("Colonne 'statement' absente : utilisation de 'clean_text' comme texte Deep Learning.")

df_model["dl_num_words"] = df_model[TEXT_DL_COL].apply(lambda x: len(str(x).split()))

display(df_model[[TEXT_DL_COL, "clean_text", LABEL_COL]].head())
display(df_model[["clean_num_words", "dl_num_words"]].describe().round(2))

## 6. Filtrage raisonnable pour le Deep Learning

On supprime uniquement :
- les textes vides ;
- les textes trop courts après nettoyage léger.

On ne supprime pas les classes minoritaires.  
Le but est d’éviter que le modèle apprenne sur des séquences presque vides.

In [ ]:
MIN_WORDS = 3

df_dl = df_model.copy()
df_dl[TEXT_DL_COL] = df_dl[TEXT_DL_COL].fillna("").astype(str).str.strip()
df_dl = df_dl[df_dl[TEXT_DL_COL] != ""].copy()
df_dl["dl_num_words"] = df_dl[TEXT_DL_COL].apply(lambda x: len(x.split()))
df_dl = df_dl[df_dl["dl_num_words"] >= MIN_WORDS].copy()

print("Dataset Deep Learning final :", df_dl.shape)
display(df_dl[LABEL_COL].value_counts())
display(df_dl["dl_num_words"].describe().round(2))

## 7. Split train / validation / test stratifié

On garde un split stratifié pour conserver la distribution des classes.

Important :
- le tokenizer doit être entraîné uniquement sur le train ;
- validation et test ne doivent pas influencer la tokenisation.

In [ ]:
X = df_dl[TEXT_DL_COL].astype(str).values
y = df_dl[LABEL_COL].astype(str).values

label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_enc,
    test_size=0.30,
    random_state=SEED,
    stratify=y_enc
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)
print("Classes :", label_encoder.classes_)

## 8. Paramètres de tokenisation adaptés au dataset

On choisit :
- `MAX_WORDS` : nombre maximal de mots dans le vocabulaire ;
- `MAX_LEN` : longueur maximale des séquences.

Au lieu de choisir arbitrairement une longueur trop grande, on utilise le 95e percentile des longueurs.  
Cela permet de couvrir la majorité des textes sans rendre l’entraînement trop lourd.

In [ ]:
train_lengths = pd.Series(X_train).apply(lambda x: len(str(x).split()))

MAX_WORDS = 50000
MAX_LEN = int(np.percentile(train_lengths, 95))

# Sécurité : éviter des séquences trop petites ou trop grandes
MAX_LEN = max(40, min(MAX_LEN, 300))

print("Longueur moyenne train :", round(train_lengths.mean(), 2))
print("Longueur médiane train :", round(train_lengths.median(), 2))
print("95e percentile :", int(np.percentile(train_lengths, 95)))
print("MAX_LEN retenu :", MAX_LEN)
print("MAX_WORDS retenu :", MAX_WORDS)

## 9. Tokenisation et padding

Le tokenizer transforme les mots en entiers.  
Le padding rend toutes les séquences de même longueur pour que le réseau neuronal puisse les traiter.

C’est la différence majeure avec TF-IDF :
- TF-IDF crée une matrice de fréquence ;
- Deep Learning crée une séquence de tokens qui conserve l’ordre des mots.

In [ ]:
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>",
    filters=""  # on garde la ponctuation déjà contrôlée dans dl_text_light
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

VOCAB_SIZE = min(MAX_WORDS, len(tokenizer.word_index) + 1)

print("VOCAB_SIZE :", VOCAB_SIZE)
print("X_train_pad shape :", X_train_pad.shape)
print("X_val_pad shape :", X_val_pad.shape)

## 10. Gestion du déséquilibre avec class weights

Le déséquilibre existe toujours dans les données réelles.  
Pour le Deep Learning, on utilise `class_weight` afin de pénaliser plus fortement les erreurs sur les classes minoritaires.

Cela évite de modifier artificiellement validation/test.

In [ ]:
class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(np.unique(y_train), class_weights_values)
}

print("Class weights :")
for idx, weight in class_weights.items():
    print(label_encoder.inverse_transform([idx])[0], ":", round(weight, 3))

## 11. Fonctions d’évaluation communes

On centralise l’évaluation pour comparer proprement les modèles.

On affiche :
- accuracy ;
- macro F1 ;
- weighted F1 ;
- classification report par classe.

Le macro F1 reste la métrique principale car les classes sont déséquilibrées.

In [ ]:
def evaluate_dl_model(model, X_eval, y_eval, model_name):
    proba = model.predict(X_eval, verbose=0)
    y_pred = np.argmax(proba, axis=1)

    acc = accuracy_score(y_eval, y_pred)
    macro = f1_score(y_eval, y_pred, average="macro")
    weighted = f1_score(y_eval, y_pred, average="weighted")

    print("\n" + "="*80)
    print("Évaluation :", model_name)
    print("="*80)
    print("Accuracy :", round(acc, 4))
    print("Macro F1 :", round(macro, 4))
    print("Weighted F1 :", round(weighted, 4))
    print("\nClassification report :")
    print(
        classification_report(
            y_eval,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0
        )
    )

    return {
        "model": model_name,
        "accuracy": acc,
        "f1_macro": macro,
        "f1_weighted": weighted
    }, y_pred

def plot_history(history, title):
    hist = pd.DataFrame(history.history)

    plt.figure(figsize=(10, 4))
    plt.plot(hist["loss"], label="train_loss")
    plt.plot(hist["val_loss"], label="val_loss")
    plt.title(title + " - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    if "accuracy" in hist.columns:
        plt.figure(figsize=(10, 4))
        plt.plot(hist["accuracy"], label="train_accuracy")
        plt.plot(hist["val_accuracy"], label="val_accuracy")
        plt.title(title + " - Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.show()

## 12. Modèle corrigé 1 — BiLSTM optimisé

Ce modèle corrige les limites du modèle `Embedding + GlobalAveragePooling`.

Différence principale :
- GlobalAveragePooling moyenne les mots ;
- BiLSTM lit la séquence dans les deux directions.

Cela permet au modèle de mieux comprendre :
- les négations ;
- l’ordre des mots ;
- les expressions émotionnelles longues.

In [ ]:
def build_bilstm_model(
    vocab_size,
    max_len,
    num_classes,
    embedding_dim=128,
    lstm_units=96,
    dropout_rate=0.4
):
    model = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            input_length=max_len
        ),
        SpatialDropout1D(0.25),
        Bidirectional(
            LSTM(
                lstm_units,
                return_sequences=True,
                dropout=0.25,
                recurrent_dropout=0.0
            )
        ),
        GlobalMaxPooling1D(),
        Dense(128, activation="relu", kernel_regularizer=l2(1e-4)),
        Dropout(dropout_rate),
        Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

num_classes = len(label_encoder.classes_)

bilstm_model = build_bilstm_model(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    num_classes=num_classes
)

bilstm_model.summary()

## 13. Entraînement du BiLSTM optimisé

On utilise :
- EarlyStopping pour éviter le sur-apprentissage ;
- ReduceLROnPlateau pour réduire le learning rate si la validation stagne ;
- class weights pour traiter le déséquilibre.

Le modèle conserve automatiquement la meilleure version selon `val_loss`.

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5
    )
]

history_bilstm = bilstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

plot_history(history_bilstm, "BiLSTM optimisé")
bilstm_result, y_pred_bilstm = evaluate_dl_model(
    bilstm_model,
    X_val_pad,
    y_val,
    "BiLSTM optimisé"
)

## 14. Modèle corrigé 2 — CNN + BiLSTM

Cette architecture ajoute une couche `Conv1D` avant le BiLSTM.

Pourquoi ?
- CNN détecte des motifs locaux : expressions courtes, mots voisins, patterns émotionnels ;
- BiLSTM apprend ensuite les relations séquentielles.

Cette combinaison peut être utile pour les textes de réseaux sociaux où certaines expressions courtes sont très discriminantes.

In [ ]:
def build_cnn_bilstm_model(
    vocab_size,
    max_len,
    num_classes,
    embedding_dim=128,
    filters=128,
    lstm_units=80,
    dropout_rate=0.4
):
    model = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            input_length=max_len
        ),
        SpatialDropout1D(0.25),
        Conv1D(filters=filters, kernel_size=5, activation="relu"),
        Bidirectional(
            LSTM(
                lstm_units,
                return_sequences=True,
                dropout=0.25
            )
        ),
        GlobalMaxPooling1D(),
        Dense(128, activation="relu", kernel_regularizer=l2(1e-4)),
        Dropout(dropout_rate),
        Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

cnn_bilstm_model = build_cnn_bilstm_model(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    num_classes=num_classes
)

cnn_bilstm_model.summary()

## 15. Entraînement du CNN + BiLSTM

On garde les mêmes règles d’entraînement que pour le BiLSTM afin de comparer les modèles de manière équitable.

In [ ]:
history_cnn_bilstm = cnn_bilstm_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

plot_history(history_cnn_bilstm, "CNN + BiLSTM")
cnn_bilstm_result, y_pred_cnn_bilstm = evaluate_dl_model(
    cnn_bilstm_model,
    X_val_pad,
    y_val,
    "CNN + BiLSTM"
)

## 16. Comparaison des modèles Deep Learning

On compare les modèles entraînés dans ce notebook.

L’objectif est de vérifier si :
- BiLSTM améliore le modèle simple ;
- CNN + BiLSTM améliore encore les classes faibles ;
- le macro F1 progresse par rapport aux modèles classiques.

In [ ]:
dl_results = pd.DataFrame([
    bilstm_result,
    cnn_bilstm_result
])

display(
    dl_results
    .sort_values(by="f1_macro", ascending=False)
    .round(4)
)

## 17. Matrice de confusion du meilleur modèle Deep Learning

La matrice de confusion permet d’identifier les erreurs précises.

On veut surtout vérifier :
- quelles classes sont confondues avec `Depression` ;
- si `Stress` est confondu avec `Anxiety` ;
- si `Personality disorder` reste mal détectée.

In [ ]:
best_dl_name = dl_results.sort_values(by="f1_macro", ascending=False).iloc[0]["model"]

if best_dl_name == "BiLSTM optimisé":
    best_dl_model = bilstm_model
    y_pred_best_dl = y_pred_bilstm
else:
    best_dl_model = cnn_bilstm_model
    y_pred_best_dl = y_pred_cnn_bilstm

print("Meilleur modèle Deep Learning :", best_dl_name)

plt.figure(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_val,
    y_pred_best_dl,
    display_labels=label_encoder.classes_,
    xticks_rotation=45,
    cmap="Blues"
)
plt.title("Matrice de confusion - Meilleur modèle Deep Learning")
plt.show()

## 18. Analyse des erreurs

On affiche quelques exemples mal classés.

Cette étape est très importante pour expliquer au jury pourquoi certaines classes restent difficiles malgré l’amélioration du modèle.

In [ ]:
errors_df = pd.DataFrame({
    "text": X_val,
    "true_label": label_encoder.inverse_transform(y_val),
    "predicted_label": label_encoder.inverse_transform(y_pred_best_dl)
})

errors_df = errors_df[errors_df["true_label"] != errors_df["predicted_label"]].copy()

print("Nombre d'erreurs validation :", len(errors_df))
display(errors_df.head(20))

## 19. Sauvegarde du meilleur modèle et du tokenizer

On sauvegarde :
- le modèle Keras ;
- le tokenizer ;
- le label encoder ;
- les paramètres de séquence.

Ces fichiers seront utiles pour le déploiement ou une future interface.

In [ ]:
import pickle

OUTPUT_DIR = Path("models_deep_learning")
OUTPUT_DIR.mkdir(exist_ok=True)

best_dl_model.save(OUTPUT_DIR / "best_deep_learning_model.keras")

with open(OUTPUT_DIR / "tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open(OUTPUT_DIR / "label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

with open(OUTPUT_DIR / "sequence_config.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "MAX_WORDS": MAX_WORDS,
            "MAX_LEN": MAX_LEN,
            "VOCAB_SIZE": VOCAB_SIZE,
            "TEXT_COLUMN_USED": TEXT_DL_COL,
            "best_model": best_dl_name
        },
        f,
        indent=4
    )

print("Modèle et objets sauvegardés dans :", OUTPUT_DIR)

## 20. Conclusion attendue

À la fin de ce notebook, deux cas sont possibles.

### Cas 1 — Le BiLSTM améliore clairement le score

Cela signifie que le problème venait principalement des limites des représentations classiques.  
Dans ce cas, le Deep Learning devient la direction principale du projet.

### Cas 2 — Le score reste proche de 0.69–0.72

Cela signifie que le problème vient probablement davantage de la structure du dataset :
- labels ambigus ;
- classes faibles ;
- chevauchement émotionnel ;
- bruit d’annotation.

Dans ce cas, la suite logique serait :
1. tester une fusion intelligente des classes faibles ;
2. tester un modèle transformer léger comme DistilBERT ;
3. ou reformuler le problème en détection binaire/multiniveau : normal / risque modéré / risque élevé.

Dans tous les cas, ce notebook permet de justifier scientifiquement pourquoi le projet passe des méthodes classiques au Deep Learning.